**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kauvar-2025-001326"], check=True)
    os.chdir("/content/encore/entries/kauvar-2025-001326")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing the behavior of Kauvar, Richman, Liu et al. (2025) from dandiset 001347

Paper. Kauvar IV, Richman EB, Liu TX, et al. (2025). Conserved brain-wide emergence of emotional
response from sensory experience in humans and mice. Science 388, eadt3971.
[doi:10.1126/science.adt3971](https://doi.org/10.1126/science.adt3971). The claims file
(`claims/001326/10.1126-science.adt3971.yaml`) was written from the full text with its supplement
(table S3).

Dataset. [DANDI:001347](https://dandiarchive.org/dandiset/001347/0.250528.0702), version
0.250528.0702. The paper's data statement points to this dandiset for the human and mouse behavior
without naming a version; 0.250528.0702 is its only published version, so it is pinned. The mouse
Neuropixels sessions are in dandiset 001326 and are tested in the entry's other notebook; this
notebook reads 001326 (version 0.250528.1957) only for the stimulus tables, to link the recorded
mice's eye closure files to those sessions.

What 001347 holds that is used here. The eye closure traces of the behavior-rig mice
(`sub-mouse`, one file per experiment id), the eye closure of the ten Neuropixels-recorded mice
(`sub-m<mouse>-<rec>`), and the per-trial eye closure of the four non-implanted participants
(`sub-h009` to `sub-h013`, P50 study). The iEEG files (`sub-SD*`, `sub-H`) are not analysed here; the data check lists the datasets every one of them holds, to look for the data of the blocked claims, and the derived `sub-H` files are analysed in `reproduce_kauvar_2025_ieeg.ipynb`.

Outside input. The `sub-mouse` files carry an experiment id and nothing else: no drug, dose,
pressure or injection label. The session-to-condition lists (and the trial selections) come from the
paper's behavior code, Zenodo [doi:10.5281/zenodo.14920570](https://doi.org/10.5281/zenodo.14920570)
(`mouse_eyepuff/scripts/fig1_*.py`). These only name the experiments the dataset already holds (a
`with_labels` input). The recorded-mouse files are matched to the 001326 sessions by the date and
mouse in `convert_npx_mouse_airpuff_to_nwb.py` of the same deposit, and the match is tested below
on the puff series timing. The human files hold `pre_trials` and `post_trials`; the same code
(`human_ieeg/scripts/analyze_airpuff_p50.py`) labels them saline and ketamine.

No step draws random numbers; the curve fits are deterministic.

| Claim | Section | Status |
|---|---|---|
| fig4b-recorded-eye-closure-ketamine | Recorded mice | tested (10 of the 13 sessions are in 001347) |
| fig5b-recorded-accumulation | Recorded mice | tested (of the 11 sessions, 10, 8 or 9 are in 001347, by reading) |
| figs34c-tone-no-eye-closure | Recorded mice | tested (6 of the 7 tone sessions are in 001347) |
| fig1k-mouse-ketamine-late-closure | Behavior-rig mice | tested |
| fig1l-dissociatives-preinfusion | Behavior-rig mice | tested |
| figs2bc-mouse-early-late | Behavior-rig mice | tested |
| fig1m-accumulation-persistence | Behavior-rig mice | tested |
| figs2g-scaling-puff-number | Behavior-rig mice | tested |
| figs2h-dose-response | Behavior-rig mice | tested |
| fig1p-anesthesia-abolishes-both | Behavior-rig mice | tested |
| figs5-other-drugs | Behavior-rig mice | tested |
| figs6-vpm-muscimol | Behavior-rig mice | tested |
| fig5hi-behavior-model-doses | Behavior-rig mice | tested (depends on fig5g-tau-persist-model, other notebook) |
| figs4-reward-licking | | blocked: no lick data in 001347 |
| fig1e-human-ketamine-late-closure | Humans | tested |
| figs1b-human-early-closure-unchanged | Humans | tested |
| fig1f-human-reports | | blocked: no participant reports in 001347 |
| fig2d-cadss-ketamine | | blocked: no CADSS scores in 001347 |
| figs14g-cadss-correlation | | blocked: no CADSS scores in 001347 |
| link-001347-condition-labels | Data check | tested |
| link-001347-recorded-sessions | Data check | tested |

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import scipy.stats

import behavior as B

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
np.set_printoptions(precision=3, suppress=True, linewidth=180)

started = time.time()
urls = B.asset_urls()
ids = sorted(set(sum(B.EXPERIMENTS.values(), [])))
mouse = B.load_mouse(ids, urls)
npx = B.load_npx(urls)
p50 = B.load_p50(urls)
stim = B.stim_001326(list(B.NPX_SESSIONS.values()))
print(f"dandiset {B.DANDISET} version {B.VERSION}: {len(urls)} files; loaded {len(mouse)} behavior-rig "
      f"experiments, {len(npx)} recorded-mouse files, {len(p50)} human files, and {len(stim)} stimulus "
      f"tables of dandiset {B.DANDISET_NPX} version {B.VERSION_NPX}, in {time.time() - started:.0f} s")


def ci(x):
    # mean and 95% interval (1.96 s.e.m., population s.d. as analysis.py; sem with ddof=1 also shown)
    x = np.asarray(x, float)
    m, s0, s1 = x.mean(), x.std() / np.sqrt(len(x)), scipy.stats.sem(x)
    return f"{m:.3f} [{m - 1.96 * s0:.3f}, {m + 1.96 * s0:.3f}] (ddof=1: [{m - 1.96 * s1:.3f}, {m + 1.96 * s1:.3f}]) n={len(x)}"

## Data check

The checks below compare the files with what the code and the paper say about them: which
experiment ids exist, trial counts, sampling rates, value ranges, whether the puffs fall where the
trial structure puts them (against a shifted control), recordings deposited twice, the link of
the recorded-mouse files to dandiset 001326, and (revised after verification) the datasets of every file
other than the behavior-rig mice, searched for the lick, report and CADSS data of the blocked claims.

In [ ]:
import collections, hashlib
kinds = collections.Counter(p.split("/")[0] if not p.startswith("sub-mouse") else "sub-mouse" for p in urls)
print("files by folder:", dict(sorted(kinds.items())))

# ---- behavior-rig mice
present = {int(p.split("ses-")[1].split("_")[0]) for p in urls if p.startswith("sub-mouse/")}
named = set(ids)
other = set(sum(B.EXPERIMENTS_OTHER.values(), []))
print(f"\nsub-mouse experiment ids: {len(present)} in 001347, {len(named)} in the code lists the claims use, {len(other - named)} more in "
      f"code lists of other panels ({', '.join(B.EXPERIMENTS_OTHER)}); named but absent {sorted((named | other) - present)}, "
      f"present but in no code list {sorted(present - named - other)}")
meta = pd.DataFrame({i: m for i, (_, m) in mouse.items()}).T
print("subject descriptions:", meta.description.unique(), "| units:", meta.unit.unique(), "| rate attribute (Hz):", meta.rate.unique())
print("subject_id equals the experiment id in every file:", all(str(i) == meta.loc[i, "subject_id"] for i in mouse))
rows = []
for c, lst in B.EXPERIMENTS.items():
    nf = [mouse[i][1]["n_frames"] for i in lst]
    rows.append({"condition": c, "n_mice": len(lst), "ids": lst, "trials in file": sorted({n // B.FRAMES_PER_TRIAL for n in nf}),
                 "trials used by code": B.NTRIALS[c], "pre trials": f"{B.WHICH_PRE[c][0]}-{B.WHICH_PRE[c][-1]}",
                 "infusion trials": f"{B.WHICH_POST[c][0]}-{B.WHICH_POST[c][-1]}"})
print(pd.DataFrame(rows).to_string(index=False))
allx = np.concatenate([x for x, _ in mouse.values()])
print(f"eye closure values: min {np.nanmin(allx):.3f}, 1st pct {np.nanpercentile(allx, 1):.3f}, median {np.nanmedian(allx):.3f}, "
      f"99th pct {np.nanpercentile(allx, 99):.3f}, max {np.nanmax(allx):.3f}, NaN {int(np.isnan(allx).sum())} of {allx.size} "
      "(the code calls it a correlation with an image of the closed eye; no unit beyond the rate)")
# puff timing: closure 0.1-0.2 s after each nominal puff minus the 0.5 s before it, against the grid shifted by 1.5 s
def jump(shift):
    v = []
    for i, (x, m) in mouse.items():
        nt = m["n_frames"] // B.FRAMES_PER_TRIAL
        tr = x[:nt * B.FRAMES_PER_TRIAL].reshape(nt, -1)
        for k in range(8):
            p = int(B.FPS * (B.T_FIRST + 3 * k + shift))
            v.append(np.nanmean(tr[:, p + 20:p + 40]) - np.nanmean(tr[:, p - 100:p]))
    return np.array(v)
for s in (0.0, 1.5):
    j = jump(s)
    print(f"puff grid shifted by {s} s: closure jump 0.1-0.2 s after minus 0.5 s before, mean {j.mean():.3f}, "
          f"min {j.min():.3f}, fraction > 0.05: {np.mean(j > 0.05):.2f} (n = {len(j)} experiment x puff)")
# recordings deposited twice
hashes = collections.defaultdict(list)
for i, (x, _) in mouse.items():
    hashes[hashlib.md5(np.nan_to_num(x[:B.FRAMES_PER_TRIAL * 20]).tobytes()).hexdigest()].append(i)
dups = [v for v in hashes.values() if len(v) > 1]
print("identical first-20-trial traces across files:", dups if dups else "none")
both = sorted(set(B.EXPERIMENTS["ket35"]) & set(B.EXPERIMENTS["28 PSI"]))
print(f"the code uses the same files for 'ket35' and '28 PSI' (the 28 PSI preinfusion trials of the ket35 experiments): {both}")
# every dataset in one sub-mouse file (metadata only), computed from the file (revised after verification)
first_mouse = sorted(p for p in urls if p.startswith("sub-mouse/"))[0]
mobj = {n: sh for n, sh in B.file_objects(urls[first_mouse]).items() if not n.startswith(("specifications", "general/"))}
print(f"datasets in {first_mouse} (outside general/):", mobj)

# ---- recorded mice
print("\nrecorded-mouse files (sub-m<mouse>-<rec>):")
rows = []
for name, d in npx.items():
    x = d["eyeclosure"]
    rows.append({"file subject": d["subject_id"], "001326 session (code)": B.NPX_SESSIONS[name], "fps": d["fps"], "rate attr": d["rate"],
                 "samples": len(x), "duration_s": round(len(x) / d["fps"]), "series events": len(d["events_on"]),
                 "by_trial": d["by_trial_shape"], "by_puff": d["by_puff_shape"], "ket_trial": d["ket_trial"],
                 "preket": f"{d['trial_range_preket'][0]}-{d['trial_range_preket'][-1]}",
                 "ket": f"{d['trial_range_ket'][0]}-{d['trial_range_ket'][-1]}",
                 "min": np.nanmin(x), "max": np.nanmax(x), "NaN": int(np.isnan(x).sum())})
print(pd.DataFrame(rows).to_string(index=False))
print("by_puff rows / 8 against the series events:", {n: (d["by_puff_shape"][0] // 8, len(d["events_on"])) for n, d in npx.items()})
for shift in (5.0, 3.5):
    j = []
    for d in npx.values():
        x, f = d["eyeclosure"], d["fps"]
        for e in d["events_on"]:
            for k in range(8):
                p = int(e + (shift + 3 * k) * f)
                if p + 20 < len(x):
                    j.append(np.nanmean(x[p + 10:p + 20]) - np.nanmean(x[p - 50:p]))
    j = np.array(j)
    print(f"puff k at events-on + {shift} + 3k s: closure 0.1-0.2 s after minus 0.5 s before, mean {np.nanmean(j):.3f}, "
          f"fraction > 0.05: {np.nanmean(j > 0.05):.2f} (n = {len(j)})")
same_day = [("m11208-959", "m11210-960")]
for a, b in same_day:
    n = min(len(npx[a]["eyeclosure"]), len(npx[b]["eyeclosure"]))
    xa, xb = npx[a]["eyeclosure"][:n], npx[b]["eyeclosure"][:n]
    ok = ~np.isnan(xa) & ~np.isnan(xb)
    print(f"{a} and {b} (same date): correlation of the traces {np.corrcoef(xa[ok], xb[ok])[0, 1]:.3f}, identical: {np.array_equal(xa, xb)}")

# ---- link to 001326
print("\nlink to dandiset 001326 (series onsets: 001326 time = a * video time + b over the first n series):")
links = {}
rows = []
for name, d in npx.items():
    s = B.NPX_SESSIONS[name]
    links[name] = L = B.link_clock(d, stim[s])
    rows.append({"file": name, "001326 session": s, "video series": L["n_video"], "001326 series": L["n_001326"],
                 "matched": L["n_matched"], "slope a": round(L["a"], 6), "max |residual| s": round(L["max_abs_resid_s"], 4),
                 "tones in 001326": len(stim[s].get("tone", []))})
print(pd.DataFrame(rows).to_string(index=False))
all1326 = ["sub-0_ses-20230901", "sub-1_ses-20230218", "sub-1_ses-20230226", "sub-2_ses-20230902", "sub-3_ses-20230218",
           "sub-3_ses-20230225", "sub-4_ses-20230907", "sub-5_ses-20230706", "sub-5_ses-20230822", "sub-6_ses-20230823",
           "sub-7_ses-20220831", "sub-8_ses-20220228", "sub-9_ses-20230908"]
print("001326 sessions with no file in 001347:", sorted(set(all1326) - set(B.NPX_SESSIONS.values())))

# ---- humans
print("\nhuman P50 files:")
for s, (pre, post, tt) in p50.items():
    dt = np.diff(tt)
    pj = np.nanmean(pre[:, (tt > 0.1) & (tt < 0.2)]) - np.nanmean(pre[:, (tt > -0.5) & (tt < 0)])
    print(f"{s}: pre_trials {pre.shape}, post_trials {post.shape}, tt {tt[0]:.3f} to {tt[-1]:.3f} s step {dt.mean():.4f} s "
          f"({1 / dt.mean():.0f} Hz), values {np.nanmin([pre.min(), post.min()]):.3f} to {np.nanmax([pre.max(), post.max()]):.3f}, "
          f"NaN {int(np.isnan(pre).sum() + np.isnan(post).sum())}, closure jump at t = 0 (0.1-0.2 s minus -0.5-0 s) {pj:.3f}")
print("labels in the human files: only 'pre_trials_data' and 'post_trials_data'; the code takes them as saline and ketamine")

# ---- what is not in 001347: every file except the behavior-rig mice (iEEG ecephys and behavior, the
# derived human files, the human P50 files, the recorded mice) and one behavior-rig file, searched by
# dataset name (metadata only)
search = [p for p in sorted(urls) if not p.startswith("sub-mouse/")] + [first_mouse]
objs = dict(zip(search, B._pmap(lambda p: B.file_objects(urls[p]), search)))
words = ("cadss", "lick", "report", "rating", "questionnaire", "survey", "score", "reward", "blink")
hits = {p: [n for n in o if any(w in n.lower() for w in words)] for p, o in objs.items()}
hits = {p: v for p, v in hits.items() if v}
print(f"\n{len(search)} files searched for datasets named like {words}:", hits if hits else "none found")
kinds = collections.defaultdict(set)
for p, o in objs.items():
    kind = ("sub-mouse" if p.startswith("sub-mouse/") else "sub-H" if p.startswith("sub-H/") else "sub-SD ecephys" if p.endswith("_ecephys.nwb") else "sub-SD behavior" if p.startswith("sub-SD")
            else "sub-h P50" if p.startswith("sub-h") else "sub-m recorded mice")
    kinds[kind] |= {"/".join(n.split("/")[:2]) for n in o if not n.startswith(("specifications", "general/"))}
for kind, groups in sorted(kinds.items()):
    print(f"  {kind}: top-level groups {sorted(groups)[:40]}{' ...' if len(groups) > 40 else ''}")


## Recorded mice

The files `sub-m<mouse>-<rec>` hold the eye closure of the Neuropixels-recorded mice (100 Hz). The
code names each as `<date>_m<mouse>_<rec>`, and the date and mouse give one 001326 session. The data
check shows the link holds for the ten files: the onset of every puff series in the video falls on
the matching 001326 series onset within 14 ms under one linear clock map. 001326 has two more series
at the end than the video in nine sessions (the video stops first), and `m94-549` has 79 series
events against 40 series in 001326 (the first 40 match). Three 001326 sessions have no file:
sub-3_ses-20230225 and both sub-5 sessions, so the paper's 13 sessions are 10 here, and its 11
(Fig. 5) are at most 10.

Method (analysis.py `make_puffseries_behavior_plot`, lines 38-90, Zenodo doi:10.5281/zenodo.14876694):
puff 1 at `events-on` + 5 s, puffs 3 s apart, each puff cut from 1 s before to 2 s after (5 s more
after puff 8), each series minus the mean of the 1 s before its first puff, Pre series 1 to 20,
Ketamine series 23 to 32, mean over series per session, each condition's mean trace divided by its
maximum (the code). Late closure is the mean over 0.3 to 0.8 s after a puff.

### fig4b-recorded-eye-closure-ketamine

> The recorded mice showed affective eye closure after the first puff of each series, and ketamine reduced it.

Criterion (claims file): eye closure of the recorded sessions, baseline-subtracted with the 1 s before
the puff (table S3), first puff of each series, Pre series 1 to 20 against Ketamine series 23 to 32
(code; Methods-literal 22 to 32 as a variant), with and without the code's normalization by each
trace's maximum (each condition's mean trace divided by its own maximum; dividing both by the Pre
maximum is a further variant); late closure 0.3 to 0.8 s; reproduced when the across-session mean of Pre minus
Ketamine has a 95% interval (1.96 s.e.m.) above zero. The file's own `trial_range_ket` (series 23
to 30, 8 series, matching the 64 Ketamine puffs of table S3) is shown as a further variant.

In [ ]:
npx_first = {n: B.npx_series(d, first_only=True) for n, d in npx.items()}
npx_all = {n: B.npx_series(d) for n, d in npx.items()}
readings = {"code (Pre 1-20, Ket 23-32)": (np.arange(20), np.arange(22, 32)),
            "Methods (Pre 1-20, Ket 22-32)": (np.arange(20), np.arange(21, 32))}
rows = []
for n, d in npx.items():
    S = npx_first[n]
    rd = dict(readings)
    rd["file ranges (Pre 1-20, Ket 23-30)"] = (d["trial_range_preket"], d["trial_range_ket"])
    for lab, (pre, ket) in rd.items():
        mp, mk = np.nanmean(S[pre], 0), np.nanmean(S[ket], 0)
        for norm in ("raw", "max-normalized", "both divided by the Pre maximum"):
            a, b = {"raw": (mp, mk), "max-normalized": (mp / np.nanmax(mp), mk / np.nanmax(mk)),
                    "both divided by the Pre maximum": (mp / np.nanmax(mp), mk / np.nanmax(mp))}[norm]
            rows.append({"session": B.NPX_SESSIONS[n], "reading": lab, "norm": norm,
                         "pre_late": B.npx_late(a, 0), "ket_late": B.npx_late(b, 0)})
df4 = pd.DataFrame(rows)
df4["pre_minus_ket"] = df4.pre_late - df4.ket_late
print("series per session:", {B.NPX_SESSIONS[n]: len(s) for n, s in npx_first.items()})
print(df4[(df4.norm == "raw") & (df4.reading.str.startswith("code"))].round(3).to_string(index=False))
print(df4[(df4.norm != "raw") & (df4.reading.str.startswith("code"))].round(3).to_string(index=False))
for (lab, norm), g in df4.groupby(["reading", "norm"], sort=False):
    t, p = scipy.stats.ttest_rel(g.pre_late, g.ket_late)
    print(f"\n{lab}, {norm}: Pre late {ci(g.pre_late)}\n   Ketamine late {ci(g.ket_late)}\n   Pre - Ketamine {ci(g.pre_minus_ket)}"
          f"  (paired t = {t:.2f}, p = {p:.4f}, not a test the paper reports)")

**Result: partially reproduced.** With each condition's mean trace divided by its own maximum, as analysis.py make_puffseries_behavior_plot does, Pre minus Ketamine late closure is 0.308 [0.243, 0.373] across the 10 sessions that have a file (code epochs; 0.311 and 0.312 for the other epoch readings). Without normalization it is 0.026 [-0.030, 0.082] (Pre 0.174, Ketamine 0.148), and with both conditions divided by the Pre maximum -0.033 [-0.285, 0.219]: those intervals include zero. The criterion shows the trace with and without the code's normalization and does not choose; the reduction holds under the code's normalization only, so it is present in part. Three of the 13 sessions have no file in DANDI:001347.

### fig5b-recorded-accumulation

> In the recorded mice, eye closure accumulated across each puff series, and ketamine abolished the accumulation.

Criterion (claims file): 11 sessions from 9 mice, the excluded animal open; eye closure
baseline-subtracted with the 1 s before each series; accumulation is late closure (0.3 to 0.8 s) after
puff 8 minus after puff 1; reproduced when the 95% interval (1.96 s.e.m. across sessions) is above zero
for Pre and includes zero or lies below the Pre mean for Ketamine (series 23 to 32).

Readings of the excluded animal: in 001326 removing sub-1, sub-3 or sub-5 leaves 11 sessions from 9
mice. In 001347 sub-5 has no file, so excluding sub-5 leaves 10 sessions (sub-3_ses-20230225 is
also absent), excluding sub-1 leaves 8, excluding sub-3 leaves 9. Each is shown, raw and with the
code's normalization by the maximum of each condition's mean trace; the Methods-literal Ketamine
series 22 to 32 and the file's 23 to 30 are variants.

In [ ]:
rows = []
for n, d in npx.items():
    S = npx_all[n]
    rd = dict(readings)
    rd["file ranges (Pre 1-20, Ket 23-30)"] = (d["trial_range_preket"], d["trial_range_ket"])
    for lab, (pre, ket) in rd.items():
        mp, mk = np.nanmean(S[pre], 0), np.nanmean(S[ket], 0)
        for norm in ("raw", "max-normalized", "both divided by the Pre maximum"):
            a, b = {"raw": (mp, mk), "max-normalized": (mp / np.nanmax(mp), mk / np.nanmax(mk)),
                    "both divided by the Pre maximum": (mp / np.nanmax(mp), mk / np.nanmax(mp))}[norm]
            rows.append({"session": B.NPX_SESSIONS[n], "reading": lab, "norm": norm,
                         "pre_p1": B.npx_late(a, 0), "pre_p8": B.npx_late(a, 7),
                         "ket_p1": B.npx_late(b, 0), "ket_p8": B.npx_late(b, 7)})
df5 = pd.DataFrame(rows)
df5["pre_acc"] = df5.pre_p8 - df5.pre_p1
df5["ket_acc"] = df5.ket_p8 - df5.ket_p1
print(df5[df5.reading.str.startswith("code")].round(3).to_string(index=False))
exclusions = {"exclude sub-5 (no file; 10 sessions)": "sub-5_", "exclude sub-1 (8 sessions)": "sub-1_", "exclude sub-3 (9 sessions)": "sub-3_"}
for (lab, norm), g in df5.groupby(["reading", "norm"], sort=False):
    for elab, pref in exclusions.items():
        h = g[~g.session.str.startswith(pref)]
        pm = h.pre_acc.mean()
        s = h.ket_acc.std(ddof=0) / np.sqrt(len(h))
        lo, hi = h.ket_acc.mean() - 1.96 * s, h.ket_acc.mean() + 1.96 * s
        print(f"\n{lab}, {norm}, {elab}:\n   Pre accumulation {ci(h.pre_acc)}\n   Ketamine accumulation {ci(h.ket_acc)}"
              f"\n   Ketamine interval includes 0: {lo <= 0 <= hi}; lies below the Pre mean ({pm:.3f}): {hi < pm}")

**Result: partially reproduced.** Pre accumulation is above zero in every reading (code normalization, without sub-5: 0.200 [0.122, 0.279]; unnormalized 0.058 [0.031, 0.085]). With the code's per-condition maximum normalization the Ketamine interval lies below the Pre mean under every exclusion (0.074 [0.032, 0.116]; 0.085 and 0.076 without sub-1 and sub-3). Without normalization, which is what the criterion describes (baseline-subtracted closure), the Ketamine interval (0.040 [0.012, 0.067]) neither includes zero nor lies below the Pre mean, and with both conditions divided by the Pre maximum it does not either (0.134 [0.056, 0.211]). Accumulation holds; its loss under ketamine holds only under the code's normalization. At most 10 sessions are available (sub-5 has no file).

### figs34c-tone-no-eye-closure

> Mice did not close their eyes appreciably to a series of auditory tones, before or during ketamine.

Criterion (claims file): eye closure across the tone series, 7 sessions from 7 mice, Pre 10 series,
Ketamine 5 (table S3); reproduced when the 95% interval across sessions (1.96 s.e.m.) of the mean
closure from 0.3 to 0.8 s after each tone, averaged over the eight tones, lies below the eyepuff
value of the same sessions and below 0.1 of full closure.

Method. The recorded-mouse files hold no tone events. The tone onsets are taken from the 001326
stimulus table and placed on the video by the clock map of the data check. Tone series are cut as
puff series (1 s before to 2 s after each tone, the series minus the 1 s before its first tone).
Pre is the first 10 tone series before ketamine onset; two splits are shown, the code's ketamine
onset (30 s after the last puff of series 20, analysis.py `get_windowed_tone_psth`) and the
infusion time of the 001326 file. Ketamine is the first 5 series after the split (table S3), 10 as
a variant (the code's tone analyses). The eyepuff value is the same measure over the 8 puffs of
Pre series 1 to 20 and Ketamine series 23 to 32. The eye closure values run from 0 to 1 (data
check), read as the fraction of full closure; values divided by the maximum of the session's Pre
puff trace (the code's normalization for Fig. 4B) are shown as a variant. The 7 sessions of table S3
are the 9 tone sessions of 001326 without sub-5; 6 of them have a file here (sub-3_ses-20230225 is
absent).

In [ ]:
def mean8(tr):
    return np.mean([B.npx_late(tr, k) for k in range(8)])

tone_rows = []
for n, d in npx.items():
    s = B.NPX_SESSIONS[n]
    st = stim[s]
    if "tone" not in st:
        continue
    t0, T = B.tone_series(d, st, links[n])
    _, groups = B.series_starts(st["puff"])
    ket_start = groups[19][-1] + 30
    S = npx_all[n]
    pp, pk = np.nanmean(S[:20], 0), np.nanmean(S[22:32], 0)
    mx = np.nanmax(pp)
    for split_lab, split in (("code ket onset", ket_start), ("file infusion time", st["ket_infusion"])):
        pre = np.where(t0 < split)[0][:10]
        post = np.where(t0 > split)[0]
        for kn in (5, 10):
            tp, tk = np.nanmean(T[pre], 0), np.nanmean(T[post[:kn]], 0)
            tone_rows.append({"session": s, "split": split_lab, "ket series": kn, "n tone series in video": len(t0),
                              "n pre": len(pre), "n ket": len(post[:kn]),
                              "tone_pre": mean8(tp), "tone_ket": mean8(tk), "puff_pre": mean8(pp), "puff_ket": mean8(pk), "puff_pre_max": mx})
dft = pd.DataFrame(tone_rows)
print(dft[(dft.split == "code ket onset") & (dft["ket series"] == 5)].round(3).to_string(index=False))
for (split_lab, kn), g in dft.groupby(["split", "ket series"], sort=False):
    print(f"\nsplit at {split_lab}, Ketamine = first {kn} tone series (sessions {len(g)}; Pre series per session {g['n pre'].tolist()})")
    for norm in ("raw", "divided by Pre puff maximum"):
        f = 1 if norm == "raw" else g.puff_pre_max
        for ph in ("pre", "ket"):
            tone = g[f"tone_{ph}"] / f
            puff = g[f"puff_{ph}"] / f
            x = np.asarray(tone, float)
            hi = x.mean() + 1.96 * x.std() / np.sqrt(len(x))
            print(f"   {norm}, {ph}: tone {ci(tone)} | puff {ci(puff)} | tone upper bound {hi:.3f} below puff mean "
                  f"{np.mean(puff):.3f}: {hi < np.mean(puff)}; below 0.1: {hi < 0.1}")

**Result: reproduced.** In the 6 tone sessions that have a file, mean closure 0.3 to 0.8 s after the tones is below zero before ketamine (raw, code split: -0.071 [-0.108, -0.034]) and near zero during it (-0.010 [-0.024, 0.004]), below the eyepuff values of the same sessions (0.157 and 0.152) and below 0.1 under every split, Ketamine count and normalization.

## Behavior-rig mice

Method (Zenodo doi:10.5281/zenodo.14920570, mouse_eyepuff/scripts and ketmouse/eye_trace_utils.py;
details and lines in the docstring of `behavior.py`). Each trial (81 s, puff 1 at 6 s, eight puffs 3 s
apart) minus the mean of its first 2 s (`prox_norm`). Early (reflexive) closure is the mean over 0.1
to 0.2 s after a puff, late (affective) closure over 0.3 to 0.8 s (Fig. 1 legend). Normalized late
closure is the per-trial late closure divided by the mean early closure over the trials of that
condition, averaged over trials (`mean_ref=True`). Drug and dose experiments use Pre trials 6 to 20
and infusion trials 23 to 37 (15 each; `which_pre = np.arange(5, 20)`, `which_post = np.arange(22, 37)`);
PSI and muscimol experiments use trials 3 to 17. Conditions are paired by position in the code's
lists (the code's pairing of mice). Tests are paired t-tests, Benjamini-Hochberg corrected over the
family the code uses.

In [ ]:
first = B.condition_table(mouse, ["sal", "ket50", "pcp20"], 0)
eighth = B.condition_table(mouse, ["sal", "ket50", "pcp20", "bup", "dzp", "kx"], 7)
print("rows (mouse x phase):", len(first), len(eighth))

### fig1k-mouse-ketamine-late-closure

> In mice, ketamine (50 mg/kg) reduced late eye closure normalized by early closure on the first puff of each series (0.26 under saline, -0.07 under ketamine).

Expected: saline 0.26 (tolerance 0.12), ketamine -0.07 (0.12); paired t-test t = 4.737, p = 0.009, 5
mice, 15 trials each (table S3). Method: first puff, infusion trials of the saline and ketamine
experiments, paired by position (fig1_drugs_first_puff.py, `ttest_rel(saline, ket)` on per-mouse means).

In [ ]:
t, p, a, b = B.paired(first, "sal", "post", "ket50", "post", "norm")
print("per mouse, saline:", a.round(3), "ketamine:", b.round(3))
print(f"saline {a.mean():.3f} +- {scipy.stats.sem(a):.3f} (s.e.m.); ketamine {b.mean():.3f} +- {scipy.stats.sem(b):.3f}")
print(f"paired t-test saline vs ketamine: t = {t:.3f}, p = {p:.4f}")
print(f"within tolerance: saline {abs(a.mean() - 0.26) <= 0.12}, ketamine {abs(b.mean() + 0.07) <= 0.12}")

**Result: reproduced.** Normalized late closure is 0.262 under saline and -0.068 under ketamine, both within tolerance, paired t = 4.737, p = 0.0091, as table S3.

### fig1l-dissociatives-preinfusion

> Compared with preinfusion, normalized late closure was unchanged by saline but reduced by ketamine (50 mg/kg) and PCP (20 mg/kg).

Expected (table S3): pre-saline 0.29 (0.16), saline 0.26 (0.12), pre-ketamine 0.37 (0.12), ketamine
-0.07 (0.12), pre-PCP 0.35 (0.2), PCP -0.03 (0.1); corrected p 0.504, 0.002, 0.041. Method: first puff,
FDR over the three comparisons (fig1_drugs.py:193-232, `first_puff_ordering = ['sal', 'ket50', 'pcp20']`
with `annotate_pvals_pre_vs_post`).

In [ ]:
exp = {"sal": (0.29, 0.16, 0.26, 0.12), "ket50": (0.37, 0.12, -0.07, 0.12), "pcp20": (0.35, 0.2, -0.03, 0.1)}
res = []
for c in ["sal", "ket50", "pcp20"]:
    t, p, a, b = B.paired(first, c, "pre", c, "post", "norm")
    res.append((c, t, p, a, b))
pc = B.fdr([r[2] for r in res])
for (c, t, p, a, b), q in zip(res, pc):
    e = exp[c]
    print(f"{c}: pre per mouse {a.round(3)}, infusion {b.round(3)}")
    print(f"   pre {a.mean():.3f} (within tol {abs(a.mean() - e[0]) <= e[1]}), infusion {b.mean():.3f} (within tol {abs(b.mean() - e[2]) <= e[3]}); "
          f"t = {t:.3f}, p = {p:.4f}, FDR p = {q:.4f}")

**Result: reproduced.** All six means are within tolerance (saline 0.286 to 0.262, ketamine 0.371 to -0.068, PCP 0.349 to -0.033), and the FDR-corrected p values are 0.504 (saline), 0.002 (ketamine) and 0.041 (PCP), as table S3.

### figs2bc-mouse-early-late

> Under ketamine the early (reflexive) closure of mice was reduced but present (0.78 to 0.45), while late (affective) closure was abolished (0.34 to -0.02).

Expected (table S3, fig. S2 B and C, "first puff of each trial", saline against ketamine): early 0.78
(0.1) and 0.45 (0.12), t = 6.854, p = 0.002; late 0.34 (0.14) and -0.02 (0.04), t = 4.951, p = 0.008.
Method: as fig1k, metrics early and late (fig1_drugs_first_puff.py loops over 'Affective' and
'Reflexive').

In [ ]:
exp = {"early": (0.78, 0.1, 0.45, 0.12), "late": (0.34, 0.14, -0.02, 0.04)}
for m in ("early", "late"):
    t, p, a, b = B.paired(first, "sal", "post", "ket50", "post", m)
    e = exp[m]
    print(f"{m}: saline per mouse {a.round(3)}, ketamine {b.round(3)}")
    print(f"   saline {a.mean():.3f} +- {scipy.stats.sem(a):.3f} (within tol {abs(a.mean() - e[0]) <= e[1]}), "
          f"ketamine {b.mean():.3f} +- {scipy.stats.sem(b):.3f} (within tol {abs(b.mean() - e[2]) <= e[3]}); t = {t:.3f}, p = {p:.4f}")
    print(f"   ketamine early closure above zero (one-sample t): {scipy.stats.ttest_1samp(b, 0)}" if m == "early" else "")

**Result: reproduced.** Early closure falls from 0.778 to 0.451 (t = 6.854, p = 0.0024) and stays above zero, and late closure falls from 0.203 to -0.025 (t = 4.951, p = 0.0078); all four means are within tolerance and the t values are those of table S3. The saline late mean (0.203) is near the lower edge of 0.34 +- 0.14.

### fig1m-accumulation-persistence

> Late eye closure accumulated across the eight puffs of a series and decayed slowly after the last puff, with a Kaplan-Meier median persistence of 15.16 s.

Criterion (claims file): preinfusion trials, 20 per mouse in 5 mice (table S3, fig. S2 E); reproduced
when the KM median is within 1.5 s of 15.16 s and the mean late closure (0.3 to 0.8 s) after the eighth
puff exceeds that after the first with non-overlapping mean +- 1.96 s.e.m. across mice. The event
ending survival is not stated in the paper; it is taken from fig1_persistence.py (see `behavior.py`):
closure falls to 1.2 times the trial's first-5-s baseline, scanned from 30.3 s into the trial, censored
at 50 s. That script uses the saline experiments and all 40 trials (200 trials), while the legend and
table say 100 trials, 20 per mouse; both are shown, with the saline-infusion trials 21 to 40 as a
further reading. Accumulation uses the saline experiments' preinfusion trials 1 to 20 (and the code's
Pre trials 6 to 20 as a variant).

In [ ]:
for lab, trials in (("table S3: saline experiments, preinfusion trials 1-20", np.arange(20)),
                    ("code: saline experiments, all 40 trials", np.arange(40)),
                    ("saline experiments, infusion trials 21-40", np.arange(20, 40))):
    T, E, med = [], [], []
    for i in B.EXPERIMENTS["sal"]:
        t, e = B.survival_times(mouse[i][0], 40, trials)
        T.append(t); E.append(e); med.append(B.km_median(t, e))
    T, E = np.concatenate(T), np.concatenate(E)
    m = B.km_median(T, E)
    print(f"{lab}: {len(T)} trials, {int(E.sum())} events, {int(len(E) - E.sum())} censored; KM median {m:.2f} s from 30.3 s "
          f"({m + 3.3:.2f} s from the eighth puff onset); within 1.5 s of 15.16: {abs(m - 15.16) <= 1.5}; per-mouse medians {np.round(med, 2)}")
for lab, trials in (("preinfusion trials 1-20", np.arange(20)), ("code Pre trials 6-20", np.arange(5, 20))):
    p1, p8 = [], []
    for i in B.EXPERIMENTS["sal"]:
        m = B.by_trial(mouse[i][0], 40)[trials]
        p1.append(B.puff_measures(m, 0)["late"]); p8.append(B.puff_measures(m, 7)["late"])
    p1, p8 = np.array(p1), np.array(p8)
    s1, s8 = p1.std(ddof=1) / np.sqrt(5), p8.std(ddof=1) / np.sqrt(5)
    print(f"\naccumulation, saline experiments, {lab}: per mouse puff 1 {p1.round(3)}, puff 8 {p8.round(3)}")
    print(f"   puff 1 {ci(p1)}; puff 8 {ci(p8)}; non-overlapping (ddof=1): {p1.mean() + 1.96 * s1 < p8.mean() - 1.96 * s8}")

**Result: partially reproduced.** Late closure accumulates: 0.244 [0.154, 0.333] after puff 1 against 0.517 [0.413, 0.621] after puff 8, non-overlapping. With the criterion's trials (preinfusion, 20 per mouse) and the code's survival event, the Kaplan-Meier median is 13.05 s, outside 15.16 +- 1.5 s; the code's own trial set (all 40 trials) gives 14.68 s, within it.

### figs2g-scaling-puff-number

> In mice, normalized late eye closure was larger after the eighth puff of a series than after the first at each of three puff pressures (8, 18 and 28 PSI), while it did not differ between pressures at the eighth puff.

Criterion (claims file): paired t-tests across 8 mice with FDR over the six comparisons, per-mouse
means of late / early, preinfusion; reproduced when the three puff 1 against puff 8 tests give corrected
p < 0.05 with puff 8 higher and the three pressure comparisons give corrected p >= 0.05. Method:
fig1_psi.py (20 trials, trials 3 to 17, `plot_puff_comparison_bar_trials` with `mean_ref=True`).

In [ ]:
exp = {("8 PSI", 0): (0.256, 0.13), ("8 PSI", 7): (0.517, 0.11), ("18 PSI", 0): (0.239, 0.063),
       ("18 PSI", 7): (0.515, 0.073), ("28 PSI", 0): (0.286, 0.13), ("28 PSI", 7): (0.447, 0.12)}
vals = {}
for c in ("8 PSI", "18 PSI", "28 PSI"):
    for k in (0, 7):
        vals[(c, k)] = np.array([B.puff_measures(B.by_trial(mouse[i][0], 20)[B.WHICH_PRE[c]], k)["norm"] for i in B.EXPERIMENTS[c]])
        m, s = B.mean_sem(vals[(c, k)])
        e = exp[(c, k)]
        print(f"{c} puff {k + 1}: per mouse {vals[(c, k)].round(3)}; {m:.3f} +- {s:.3f} (within tol {abs(m - e[0]) <= e[1]})")
labels, ts, ps = [], [], []
for c in ("8 PSI", "18 PSI", "28 PSI"):
    t, p = scipy.stats.ttest_rel(vals[(c, 0)], vals[(c, 7)]); labels.append(f"{c}: puff 1 vs puff 8"); ts.append(t); ps.append(p)
for a, b in (("8 PSI", "18 PSI"), ("8 PSI", "28 PSI"), ("18 PSI", "28 PSI")):
    t, p = scipy.stats.ttest_rel(vals[(a, 7)], vals[(b, 7)]); labels.append(f"puff 8: {a} vs {b}"); ts.append(t); ps.append(p)
for l, t, p, q in zip(labels, ts, ps, B.fdr(ps)):
    print(f"{l}: t = {t:.3f}, p = {p:.5f}, FDR p = {q:.4f}")

**Result: reproduced.** Puff 8 exceeds puff 1 at 8, 18 and 28 PSI (corrected p = 0.0038, 0.0001 and below 0.0001), the three pressure comparisons at puff 8 give corrected p = 0.98, 0.46 and 0.60, and all six means are within tolerance.

### figs2h-dose-response

> The dissociative reduction of normalized late eye closure began around 35 mg/kg ketamine and 10 mg/kg PCP; lower doses did not significantly reduce it.

Criterion (claims file): paired t-tests of preinfusion against infusion of normalized late closure
after the eighth puff at each dose, FDR-corrected under (a) all ten comparisons of the table together
(saline included) and (b) each drug apart; reproduced when, under both, ketamine 35 and 50 and PCP 10
and 20 give corrected p < 0.05 with a decrease and the lower doses corrected p >= 0.05. Expected
infusion values: ketamine 35 mg/kg 0.17 (0.14), PCP 10 mg/kg 0.2 (0.25). Method: fig1_dose.py
(ordering of ten conditions; ket35 has 8 mice). Under (b) saline is its own family of one.

In [ ]:
order = ["sal", "ket50", "ket35", "ket25", "ket13", "ket6", "pcp20", "pcp10", "pcp5", "pcp1"]
dose = B.condition_table(mouse, order, 7)
res = {}
for c in order:
    t, p, a, b = B.paired(dose, c, "pre", c, "post", "norm")
    res[c] = (t, p, a, b)
qa = dict(zip(order, B.fdr([res[c][1] for c in order])))
qb = {}
for fam in (["sal"], ["ket50", "ket35", "ket25", "ket13", "ket6"], ["pcp20", "pcp10", "pcp5", "pcp1"]):
    qb.update(dict(zip(fam, B.fdr([res[c][1] for c in fam]))))
for c in order:
    t, p, a, b = res[c]
    print(f"{c}: n = {len(a)}, pre {a.mean():.3f}, infusion {b.mean():.3f}, t = {t:.3f}, p = {p:.4f}, FDR (a) {qa[c]:.4f}, FDR (b) {qb[c]:.4f}")
print(f"ketamine 35 infusion {res['ket35'][3].mean():.3f} (within 0.14 of 0.17: {abs(res['ket35'][3].mean() - 0.17) <= 0.14}); "
      f"PCP 10 infusion {res['pcp10'][3].mean():.3f} (within 0.25 of 0.2: {abs(res['pcp10'][3].mean() - 0.2) <= 0.25})")
print("ketamine 13 mg/kg per mouse, pre:", res["ket13"][2].round(3), "infusion:", res["ket13"][3].round(3))

**Result: reproduced.** Under both FDR families, ketamine 50 and 35 mg/kg and PCP 20 and 10 mg/kg reduce late closure (corrected p 0.005 to 0.016) and the lower doses do not (ketamine 25, 13 and 6 mg/kg: 0.357, 0.357, 0.759; PCP 5 and 1 mg/kg: 0.110 and 0.180, or 0.074 and 0.108 per drug). Ketamine 35 (0.169) and PCP 10 (0.205) are within tolerance.

### fig1p-anesthesia-abolishes-both

> A general anesthetic dose of ketamine with xylazine abolished both reflexive and affective eye closure, while ketamine or PCP alone kept the reflexive blink.

Criterion (claims file): paired t-tests with FDR of reflexive (0.1 to 0.2 s) and affective (0.3 to
0.8 s) closure after the eighth puff, 5 mice, 15 trials each; reproduced when both K/X tests give
corrected p < 0.05 with K/X lower, the K/X values are within tolerance (reflexive 0.03 +- 0.04,
affective 0.02 +- 0.02; pre-K/X 0.77 +- 0.18 and 0.47 +- 0.06), and the reflexive closure under
ketamine (0.44 +- 0.12) and PCP (0.43 +- 0.02) is within tolerance. Method: fig1_drugs.py:340-392,
FDR over the six conditions of `ordering` per metric.

In [ ]:
order6 = ["sal", "ket50", "pcp20", "bup", "dzp", "kx"]
for m, lab in (("early", "reflexive"), ("late", "affective")):
    r = [B.paired(eighth, c, "pre", c, "post", m) for c in order6]
    q = B.fdr([x[1] for x in r])
    print(f"{lab} closure, eighth puff (FDR over six conditions):")
    for c, (t, p, a, b), qq in zip(order6, r, q):
        print(f"   {c}: pre {a.mean():.3f}, infusion {b.mean():.3f} (per mouse {b.round(3)}), t = {t:.4f}, p = {p:.5f}, FDR p = {qq:.4f}")
kx = eighth[eighth.cond == "kx"]
pre_r, kx_r = kx[kx.phase == "pre"].early.mean(), kx[kx.phase == "post"].early.mean()
pre_a, kx_a = kx[kx.phase == "pre"].late.mean(), kx[kx.phase == "post"].late.mean()
ket_r = eighth[(eighth.cond == "ket50") & (eighth.phase == "post")].early.mean()
pcp_r = eighth[(eighth.cond == "pcp20") & (eighth.phase == "post")].early.mean()
print(f"within tolerance: pre-K/X reflexive {pre_r:.3f} {abs(pre_r - 0.77) <= 0.18}; K/X reflexive {kx_r:.3f} {abs(kx_r - 0.03) <= 0.04}; "
      f"pre-K/X affective {pre_a:.3f} {abs(pre_a - 0.47) <= 0.06}; K/X affective {kx_a:.3f} {abs(kx_a - 0.02) <= 0.02}; "
      f"ketamine reflexive {ket_r:.3f} {abs(ket_r - 0.44) <= 0.12}; PCP reflexive {pcp_r:.3f} {abs(pcp_r - 0.43) <= 0.02}")

**Result: reproduced.** K/X lowers reflexive closure from 0.773 to 0.025 (t = 8.6017, corrected p = 0.006) and affective closure from 0.470 to 0.024 (t = 12.0602, corrected p = 0.0016), all within tolerance, while reflexive closure stays at 0.436 under ketamine and 0.428 under PCP, within tolerance.

### figs5-other-drugs

> Buprenorphine and diazepam did not reduce normalized late closure, unlike ketamine and PCP.

Criterion (claims file): paired t-tests with FDR on normalized late closure of the eighth puff;
reproduced when ketamine and PCP give corrected p < 0.05 with a decrease and buprenorphine and diazepam
corrected p >= 0.05. Method: fig1_drugs.py:394-407 corrects over `ordering_nokx` (saline, ketamine,
PCP, buprenorphine, diazepam: five comparisons); the four drugs alone and the six with K/X are shown
as variants.

In [ ]:
r = {c: B.paired(eighth, c, "pre", c, "post", "norm") for c in order6}
for lab, fam in (("code: five conditions (K/X left out)", ["sal", "ket50", "pcp20", "bup", "dzp"]),
                 ("four drugs", ["ket50", "pcp20", "bup", "dzp"]), ("six conditions", order6)):
    q = B.fdr([r[c][1] for c in fam])
    print(lab)
    for c, qq in zip(fam, q):
        t, p, a, b = r[c]
        print(f"   {c}: pre {a.mean():.3f}, infusion {b.mean():.3f}, t = {t:.4f}, p = {p:.4f}, FDR p = {qq:.4f}")
print("K/X infusion per mouse (normalized late closure divides by an early closure near zero):", r["kx"][3].round(2))

**Result: reproduced.** With the code's family of five comparisons, ketamine (corrected p = 0.0053) and PCP (0.0092) reduce normalized late closure and buprenorphine (0.934) and diazepam (0.304) do not; the four- and six-comparison families give the same pattern.

### figs6-vpm-muscimol

> Inhibiting VPM thalamus with muscimol reduced normalized affective closure (0.60 after saline injection, 0.11 after muscimol) while the fast early closure remained.

Expected (table S3, fig. S6 C): saline injection 0.60 (0.1), muscimol 0.11 (0.24); saline against
muscimol t = 3.85, corrected p = 0.0289; no injection against muscimol t = 3.79, corrected p = 0.0289.
Method: fig1_muscimol.py (20 trials, trials 3 to 17, eighth puff; `get_drug_comparison_pvals` corrects
over three comparisons, no injection vs saline, saline vs muscimol, no injection vs muscimol; the
script then also corrects the two muscimol comparisons alone, shown). The fast early closure is shown as
the early closure (0.1 to 0.2 s) and the 0 to 40 ms window of the script.

In [ ]:
mus = B.condition_table(mouse, ["intact", "musc_sal", "vpm_musc"], 7)
for c in ("intact", "musc_sal", "vpm_musc"):
    g = mus[(mus.cond == c) & (mus.phase == "pre")]
    print(f"{c}: normalized late per mouse {g.norm.to_numpy().round(3)}, mean {g.norm.mean():.3f} +- {scipy.stats.sem(g.norm):.3f}; "
          f"early {g.early.mean():.3f} +- {scipy.stats.sem(g.early):.3f}")
pairs = [("intact", "musc_sal"), ("musc_sal", "vpm_musc"), ("intact", "vpm_musc")]
r = [B.paired(mus, a, "pre", b, "pre", "norm") for a, b in pairs]
q3, q2 = B.fdr([x[1] for x in r]), B.fdr([x[1] for x in r[1:]])
for (a, b), x, qq in zip(pairs, r, q3):
    print(f"{a} vs {b}: t = {x[0]:.3f}, p = {x[1]:.4f}, FDR (three) p = {qq:.4f}")
print("FDR over the two muscimol comparisons:", q2.round(4))
sal_m = mus[(mus.cond == "musc_sal") & (mus.phase == "pre")].norm.mean()
mus_m = mus[(mus.cond == "vpm_musc") & (mus.phase == "pre")].norm.mean()
print(f"within tolerance: saline {abs(sal_m - 0.60) <= 0.1}, muscimol {abs(mus_m - 0.11) <= 0.24}")
fast = []
for c in ("intact", "musc_sal", "vpm_musc"):
    v = []
    for i in B.EXPERIMENTS[c]:
        m = B.by_trial(mouse[i][0], 20)[B.WHICH_PRE[c]]
        a0, b0 = B.win(0, (0.0, 0.04))
        v.append(np.nanmean(m[:, a0:b0]))
    fast.append(np.array(v))
    print(f"{c}: fast early closure 0-40 ms after puff 1, per mouse {np.round(v, 3)}, {ci(v)}")

**Result: reproduced.** Normalized affective closure is 0.602 after saline injection and 0.105 after muscimol, both within tolerance (t = 3.845 and 3.789, corrected p = 0.0289 for both comparisons), and the fast early closure stays (0.089 [0.056, 0.121] under muscimol against 0.075 after saline).

### fig5hi-behavior-model-doses

> Varying only the persistence timescale of the two-phase model explained much of the variation in affective eye closure across doses of ketamine and of PCP.

Criterion (claims file): fit as for fig5g-tau-persist-model on the dose series (ketamine 6, 13, 25, 35
mg/kg; PCP 1, 5, 10 mg/kg), 750 ms after each puff masked (Methods, analysis.py `filter_reflex`);
variance explained is 1 - MSE_model/MSE_null against the fit with no parameter varying; reproduced
when the variance explained is positive for each drug and the fitted persistence timescale falls with
dose. Method: analysis.py `theoretical_behavior` (model at 10 ms over 44 s from 1 s before puff 1,
masked by `filter_reflex`), bounds and starts of `compare_models`. The data the code fed to it are not
in the released code; they are taken here (assumption) as the across-mouse mean of the per-mouse mean
of the infusion trials (23 to 37), baseline-subtracted as above, 1 s before to 43 s after puff 1,
averaged to 100 Hz (Methods: "eye closure data was averaged across participants and concatenated").
Adding the 50 mg/kg ketamine and 20 mg/kg PCP experiments is a variant. This claim depends on
fig5g-tau-persist-model, tested in the entry's other notebook.

In [ ]:
for lab, fams in (("criterion doses", (("ketamine", ["ket6", "ket13", "ket25", "ket35"]), ("PCP", ["pcp1", "pcp5", "pcp10"]))),
                  ("with ketamine 50 and PCP 20", (("ketamine", ["ket6", "ket13", "ket25", "ket35", "ket50"]), ("PCP", ["pcp1", "pcp5", "pcp10", "pcp20"])))):
    for drug, conds in fams:
        data = np.hstack([B.filter_reflex(B.model_trace(mouse, c)) for c in conds])
        r = B.fit_behavior_model(data, len(conds))
        falls = bool(np.all(np.diff(r["tauP"]) < 0))
        print(f"{lab}, {drug} {conds}: variance explained {r['varexp']:.3f}; S = {r['S']:.3f}, tauB = {r['tauB']:.3f} s; "
              f"tauP by dose {np.round(r['tauP'], 3)} s (falls with every dose step: {falls}); null tauP {np.round(r['null_tauP'], 3)}")

**Result: reproduced.** Varying only tauP explains 0.917 of the variance across ketamine doses and 0.886 across PCP doses, and the fitted tauP falls with every dose step (ketamine 6 to 35 mg/kg: 24.5, 9.7, 5.1, 0.6 s; PCP 1 to 10 mg/kg: 18.4, 10.4, 2.2 s). The ketamine fit sits at the upper bound of tauB (100 s).

## Humans (four non-implanted participants)

Method: human_ieeg/scripts/analyze_airpuff_p50.py. Trials 3 to 40 (`trial_range = np.arange(2, 40)`,
"Skip 1st and 2nd trial, which sometimes didn't have a puff"), samples with 0.3 < t < 0.8 s (late)
and 0.1 < t < 0.2 s (early), per participant the mean over trials of late divided by the mean early
closure (`mean_ref=True`); `pre_trials` taken as saline and `post_trials` as ketamine (the script's
labels). No baseline is subtracted in the script.

### fig1e-human-ketamine-late-closure

> In four non-implanted participants, late eye closure after the eyepuff (0.3 to 0.8 s), normalized by early closure (0.1 to 0.2 s), was lower during ketamine (0.43) than during saline (0.71).

Criterion (claims file): per participant, the mean over trials (the first two excluded) of late
divided by early closure; reproduced when both group means are within tolerance (saline 0.71 +- 0.16,
ketamine 0.43 +- 0.24) and a paired t-test gives p < 0.05 with ketamine lower (table S3: t = 3.703,
p = 0.0342).

In [ ]:
H = {s: (B.human_measures(pre, tt), B.human_measures(post, tt)) for s, (pre, post, tt) in p50.items()}
a = np.array([H[s][0]["norm"] for s in H]); b = np.array([H[s][1]["norm"] for s in H])
print("participants:", list(H), "trials used per condition:", len(np.arange(2, 40)))
print("saline per participant:", a.round(3), "ketamine:", b.round(3))
t, p = scipy.stats.ttest_rel(a, b)
print(f"saline {a.mean():.3f} +- {scipy.stats.sem(a):.3f} (within tol {abs(a.mean() - 0.71) <= 0.16}); "
      f"ketamine {b.mean():.3f} +- {scipy.stats.sem(b):.3f} (within tol {abs(b.mean() - 0.43) <= 0.24}); paired t = {t:.3f}, p = {p:.4f}")
la = np.array([H[s][0]["late"] for s in H]); lb = np.array([H[s][1]["late"] for s in H])
print(f"late closure without normalization (fig. S1C): saline {la.mean():.3f}, ketamine {lb.mean():.3f}, {scipy.stats.ttest_rel(la, lb)}")

**Result: reproduced.** Normalized late closure is 0.706 under saline and 0.426 under ketamine, within tolerance, paired t = 3.703, p = 0.0342, as table S3.

### figs1b-human-early-closure-unchanged

> Ketamine did not change the early reflexive eye closure of the same participants (0.84 against 0.78).

Criterion (claims file): a paired t-test of per-participant mean early closure (table S3 calls it
the sum over 0.1 to 0.2 s) under saline against ketamine; reproduced when p >= 0.05, the difference
reported either way. Expected 0.84 (0.02) and 0.78 (0.08); table S3 t = 1.622, p = 0.2032. The script
takes the mean over the samples, which gives values on the table's scale; the sum is shown too.

In [ ]:
a = np.array([H[s][0]["early"] for s in H]); b = np.array([H[s][1]["early"] for s in H])
t, p = scipy.stats.ttest_rel(a, b)
print("saline per participant:", a.round(3), "ketamine:", b.round(3))
print(f"mean over 0.1-0.2 s: saline {a.mean():.3f} +- {scipy.stats.sem(a):.3f} (within tol {abs(a.mean() - 0.84) <= 0.02}), "
      f"ketamine {b.mean():.3f} +- {scipy.stats.sem(b):.3f} (within tol {abs(b.mean() - 0.78) <= 0.08}); "
      f"difference {a.mean() - b.mean():.3f}; paired t = {t:.3f}, p = {p:.4f}")
sa = np.array([H[s][0]["early_sum"] for s in H]); sb = np.array([H[s][1]["early_sum"] for s in H])
print(f"sum over the {H[list(H)[0]][0]['n_early_samples']} samples: saline {sa.mean():.3f}, ketamine {sb.mean():.3f}, {scipy.stats.ttest_rel(sa, sb)}")
print(f"total runtime {time.time() - started:.0f} s")

**Result: reproduced.** Early closure is 0.839 under saline and 0.777 under ketamine (difference 0.062), within tolerance, and the paper's paired t-test does not find a difference (t = 1.622, p = 0.2032).

## Summary

| Claim | Outcome |
|---|---|
| fig4b-recorded-eye-closure-ketamine | partially reproduced |
| fig5b-recorded-accumulation | partially reproduced |
| figs34c-tone-no-eye-closure | reproduced |
| fig1k-mouse-ketamine-late-closure | reproduced |
| fig1l-dissociatives-preinfusion | reproduced |
| figs2bc-mouse-early-late | reproduced |
| fig1m-accumulation-persistence | partially reproduced |
| figs2g-scaling-puff-number | reproduced |
| figs2h-dose-response | reproduced |
| fig1p-anesthesia-abolishes-both | reproduced |
| figs5-other-drugs | reproduced |
| figs6-vpm-muscimol | reproduced |
| fig5hi-behavior-model-doses | reproduced |
| fig1e-human-ketamine-late-closure | reproduced |
| figs1b-human-early-closure-unchanged | reproduced |
| link-001347-condition-labels | reproduced |
| link-001347-recorded-sessions | partially reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the conditions, doses and pressures of the behavior files come from the session lists of the paper's code (Zenodo doi:10.5281/zenodo.14920570), which also gives the trial selections and windows; the recorded-mouse eye closure covers 10 of the 13 sessions; the tone onsets are placed on the video clock through the fit to the 001326 puff series.

What was found about DANDI:001347: the behavior-rig files hold only an eye closure trace and an experiment id, with no drug, dose, pressure or injection label; the human P50 files label trials only as pre and post; there are no lick data, participant reports or CADSS scores in any of its files (searched by dataset name in every file other than the behavior-rig mice and in one behavior-rig file, 66 files in all); three recorded sessions of DANDI:001326 have no eye-closure file, and the recorded-mouse files carry no tone events.
